%md ## 0 · Configuration & Imports

In [0]:
import json
import logging
from datetime import datetime, timezone
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("gold_star_schema")

spark = SparkSession.builder.appName("youtube_gold_star").getOrCreate()

%md ## 1 · Pipeline Parameters

In [0]:
dbutils.widgets.text("bucket_name",    "youtube-capstone-bucket-team6", "S3 Bucket Name")
dbutils.widgets.text("catalog_name",   "youtube-capstone-catalog",      "Unity Catalog Name")
dbutils.widgets.text("run_date",       datetime.now(timezone.utc).strftime("%Y-%m-%d"), "Run Date")
dbutils.widgets.text("growth_factor",  "1.5",  "Virality Growth Factor")
dbutils.widgets.text("trending_top_n", "50",   "Top-N for Trending Rank")

BUCKET_NAME    = dbutils.widgets.get("bucket_name")
CATALOG_NAME   = dbutils.widgets.get("catalog_name")   
RUN_DATE       = dbutils.widgets.get("run_date")
GROWTH_FACTOR  = float(dbutils.widgets.get("growth_factor"))
TRENDING_TOP_N = int(dbutils.widgets.get("trending_top_n"))

BASE_PATH    = f"s3://{BUCKET_NAME}/youtube_pipeline"
SILVER_PATH  = f"{BASE_PATH}/silver"
GOLD_PATH    = f"{BASE_PATH}/gold"

GOLD_DB      = f"`{CATALOG_NAME}`.gold"

logger.info(f"Gold config → catalog={CATALOG_NAME}, run_date={RUN_DATE}, growth_factor={GROWTH_FACTOR}")

%md ## 2 · Load Silver Tables

In [0]:
df_videos = spark.read.format("delta").load(f"{SILVER_PATH}/silver_videos")
df_comments = spark.read.format("delta").load(f"{SILVER_PATH}/silver_comments")

logger.info(f"Silver videos  : {df_videos.count():,} rows")
logger.info(f"Silver comments: {df_comments.count():,} rows")

%md ## 3 · Helpers

In [0]:
def write_gold_table(df: DataFrame, table_name: str, partition_cols: list = None) -> int:
    path = f"{GOLD_PATH}/{table_name}"

    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_cols:
        writer = writer.partitionBy(*partition_cols)
    writer.save(path)

    spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {GOLD_DB}.{table_name}
        USING DELTA
        LOCATION '{path}'
    """)
    spark.sql(f"OPTIMIZE {GOLD_DB}.{table_name}")

    count = spark.read.format("delta").load(path).count()
    logger.info(f"[Gold] {table_name}: {count:,} rows → {path}")
    return count


def add_audit_cols(df: DataFrame) -> DataFrame:
    return (
        df
        .withColumn("_gold_created_at", F.current_timestamp())
        .withColumn("_run_date",         F.lit(RUN_DATE))
    )

%md ## 4 · Compute Base Metrics (used by both Fact tables)

In [0]:
SAFE_VIEWS = F.when(F.col("view_count") > 0, F.col("view_count")).otherwise(1)

df_metrics = (
    df_videos
    .withColumn("engagement_rate",   ((F.col("like_count") + F.col("comment_count")) / SAFE_VIEWS * 100).cast("double"))
    .withColumn("virality_score",    (((F.col("like_count") + F.col("comment_count")) / SAFE_VIEWS) * F.lit(GROWTH_FACTOR)).cast("double"))
    .withColumn("like_view_ratio",   (F.col("like_count")    / SAFE_VIEWS).cast("double"))
    .withColumn("comment_view_ratio",(F.col("comment_count") / SAFE_VIEWS).cast("double"))
    .withColumn("duration_minutes",  (F.col("duration_seconds") / 60).cast("double"))
    .withColumn("virality_tier",
        F.when(F.col("virality_score") >= 0.10, "Viral")
         .when(F.col("virality_score") >= 0.05, "High")
         .when(F.col("virality_score") >= 0.01, "Medium")
         .otherwise("Low"))
    .withColumn("days_since_publish", F.datediff(F.current_date(), F.col("publish_date")))
    .withColumn("view_velocity",
        F.when(F.col("days_since_publish") > 0,
               F.col("view_count") / F.col("days_since_publish")
        ).otherwise(F.col("view_count")))
)

window_trending = Window.partitionBy("run_date").orderBy(F.col("virality_score").desc())
df_metrics = df_metrics.withColumn("trending_rank", F.dense_rank().over(window_trending))

logger.info("Core metrics computed")

%md ## 5 · DIMENSION TABLES

%md ### dim_video
**Grain:** One row per `video_id`
**Purpose:** Descriptive attributes of a YouTube video — title, thumbnail, duration, tags, publish info.
No metrics here; metrics live in the Fact tables.

In [0]:
df_dim_video = (
    df_metrics
    .select(
        "video_id",           # PK
        "title",
        "description",
        "thumbnail_url",
        "duration",           # ISO-8601 original string
        "duration_seconds",
        "duration_minutes",
        "tags_clean",
        "publish_date",
        "published_at",       # full timestamp
        "publish_year",
        "publish_month",
        "publish_dow",
    )
    .dropDuplicates(["video_id"])
    .filter(F.col("video_id").isNotNull())
)

df_dim_video = add_audit_cols(df_dim_video)
dim_video_count = write_gold_table(df_dim_video, "dim_video")

print(f"dim_video: {dim_video_count:,} rows")
df_dim_video.printSchema()

dim_video: 668 rows
root
 |-- video_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- thumbnail_url: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- tags_clean: string (nullable = true)
 |-- publish_date: date (nullable = true)
 |-- published_at: timestamp (nullable = true)
 |-- publish_year: integer (nullable = true)
 |-- publish_month: integer (nullable = true)
 |-- publish_dow: integer (nullable = true)
 |-- _gold_created_at: timestamp (nullable = false)
 |-- _run_date: string (nullable = false)



%md ### dim_channel
**Grain:** One row per `channel_id`
**Purpose:** Channel-level descriptive attributes — name and ID only.
Aggregated metrics (monthly views, growth) stay in the Fact table.

In [0]:
df_dim_channel = (
    df_metrics
    .select("channel_id", "channel_title")
    .dropDuplicates(["channel_id"])
    .filter(F.col("channel_id").isNotNull())
)

df_dim_channel = add_audit_cols(df_dim_channel)
dim_channel_count = write_gold_table(df_dim_channel, "dim_channel")

print(f"dim_channel: {dim_channel_count:,} rows")

dim_channel: 649 rows


In [0]:
CATEGORY_MAP = [
    ("1",  "Film & Animation"),
    ("2",  "Autos & Vehicles"),
    ("10", "Music"),
    ("15", "Pets & Animals"),
    ("17", "Sports"),
    ("18", "Short Movies"),
    ("19", "Travel & Events"),
    ("20", "Gaming"),
    ("21", "Videoblogging"),
    ("22", "People & Blogs"),
    ("23", "Comedy"),
    ("24", "Entertainment"),
    ("25", "News & Politics"),
    ("26", "Howto & Style"),
    ("27", "Education"),
    ("28", "Science & Technology"),
    ("29", "Nonprofits & Activism"),
    ("30", "Movies"),
    ("31", "Anime/Animation"),
    ("32", "Action/Adventure"),
    ("33", "Classics"),
    ("34", "Comedy"),
    ("35", "Documentary"),
    ("36", "Drama"),
    ("37", "Family"),
    ("38", "Foreign"),
    ("39", "Horror"),
    ("40", "Sci-Fi/Fantasy"),
    ("41", "Thriller"),
    ("42", "Shorts"),
    ("43", "Shows"),
    ("44", "Trailers"),
]

cat_static_df = spark.createDataFrame(CATEGORY_MAP, schema=["category_id", "category_name"])

cat_actual_df = (
    df_metrics
    .select("category_id")
    .dropDuplicates(["category_id"])
    .filter(F.col("category_id").isNotNull())
)

df_dim_category = (
    cat_actual_df
    .join(cat_static_df, on="category_id", how="left")
    .withColumn("category_name",
        F.coalesce(F.col("category_name"), F.concat(F.lit("Category "), F.col("category_id")))
    )
)

df_dim_category = add_audit_cols(df_dim_category)
dim_category_count = write_gold_table(df_dim_category, "dim_category")

print(f"dim_category: {dim_category_count:,} rows")

dim_category: 10 rows


%md ### dim_region
**Grain:** One row per `region_code`
**Purpose:** ISO-3166 region metadata for geographic filtering.

In [0]:
REGION_MAP = [
    ("US", "United States",    "Americas"),
    ("IN", "India",            "Asia"),
    ("GB", "United Kingdom",   "Europe"),
    ("CA", "Canada",           "Americas"),
    ("AU", "Australia",        "Oceania"),
    ("DE", "Germany",          "Europe"),
    ("FR", "France",           "Europe"),
    ("JP", "Japan",            "Asia"),
    ("KR", "South Korea",      "Asia"),
    ("BR", "Brazil",           "Americas"),
    ("MX", "Mexico",           "Americas"),
    ("RU", "Russia",           "Europe/Asia"),
    ("IT", "Italy",            "Europe"),
    ("ES", "Spain",            "Europe"),
    ("NL", "Netherlands",      "Europe"),
    ("SG", "Singapore",        "Asia"),
    ("ZA", "South Africa",     "Africa"),
    ("NG", "Nigeria",          "Africa"),
    ("EG", "Egypt",            "Africa/Asia"),
    ("SA", "Saudi Arabia",     "Asia"),
    ("AE", "United Arab Emirates", "Asia"),
    ("PH", "Philippines",      "Asia"),
    ("ID", "Indonesia",        "Asia"),
    ("PK", "Pakistan",         "Asia"),
    ("TH", "Thailand",         "Asia"),
    ("VN", "Vietnam",          "Asia"),
    ("TR", "Turkey",           "Europe/Asia"),
    ("PL", "Poland",           "Europe"),
    ("SE", "Sweden",           "Europe"),
    ("CH", "Switzerland",      "Europe"),
]

region_static_df = spark.createDataFrame(
    REGION_MAP, schema=["region_code", "country_name", "continent"]
)

region_actual_df = (
    df_metrics
    .select("region_code")
    .dropDuplicates(["region_code"])
    .filter(F.col("region_code").isNotNull())
)

df_dim_region = (
    region_actual_df
    .join(region_static_df, on="region_code", how="left")
    .withColumn("country_name",
        F.coalesce(F.col("country_name"), F.col("region_code"))
    )
    .withColumn("continent",
        F.coalesce(F.col("continent"), F.lit("Unknown"))
    )
)

df_dim_region = add_audit_cols(df_dim_region)
dim_region_count = write_gold_table(df_dim_region, "dim_region")

print(f"dim_region: {dim_region_count:,} rows")

dim_region: 5 rows


%md ### dim_date
**Grain:** One row per calendar date
**Purpose:** Enables time-series analysis and date-based drill-downs in Snowflake / Streamlit.
Built from the range of publish dates actually present in Silver data.

In [0]:
from pyspark.sql.types import DateType

date_range = df_metrics.agg(
    F.min("publish_date").alias("min_date"),
    F.max("publish_date").alias("max_date"),
).collect()[0]

min_date = date_range["min_date"]
max_date = date_range["max_date"]

logger.info(f"Date dimension range: {min_date} → {max_date}")

df_dim_date = (
    spark.range(0, (max_date - min_date).days + 2)   # +2 for buffer
    .withColumn("full_date",    (F.lit(min_date.strftime("%Y-%m-%d")).cast(DateType())
                                 + F.col("id").cast("int")).cast(DateType()))
    .select(
        F.date_format(F.col("full_date"), "yyyyMMdd").cast("int").alias("date_key"),   # surrogate key
        F.col("full_date"),
        F.year("full_date")                            .alias("year"),
        F.month("full_date")                           .alias("month"),
        F.dayofmonth("full_date")                      .alias("day"),
        F.quarter("full_date")                         .alias("quarter"),
        F.dayofweek("full_date")                       .alias("day_of_week"),          # 1=Sun … 7=Sat
        F.date_format(F.col("full_date"), "EEEE")      .alias("day_name"),
        F.date_format(F.col("full_date"), "MMMM")      .alias("month_name"),
        F.weekofyear("full_date")                      .alias("week_of_year"),
        F.when(F.dayofweek("full_date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
        F.date_format(F.col("full_date"), "yyyy-MM")   .alias("year_month"),
    )
    .filter(F.col("full_date").isNotNull())
)

df_dim_date = add_audit_cols(df_dim_date)
dim_date_count = write_gold_table(df_dim_date, "dim_date")

print(f"dim_date: {dim_date_count:,} rows  ({min_date} → {max_date})")

dim_date: 26 rows  (2026-04-03 → 2026-04-27)


%md ## 6 · FACT TABLES

In [0]:
logger.info("── Building fact_video_performance ──────────────────────────────")

df_fact_video = (
    df_metrics
    .select(
        # Foreign Keys
        "video_id",
        "channel_id",
        "category_id",
        "region_code",
        F.date_format(F.col("publish_date"), "yyyyMMdd").cast("int").alias("date_key"),
        "run_date",

        # Raw Measures
        "view_count",
        "like_count",
        "comment_count",
        "duration_seconds",
        "duration_minutes",

        # Computed Measures
        "engagement_rate",      # (likes + comments) / views * 100
        "virality_score",       # ((likes + comments) / views) * growth_factor
        "like_view_ratio",      # likes / views
        "comment_view_ratio",   # comments / views
        "trending_rank",        # dense_rank() on virality_score per run_date

        #Derived Measures 
        "virality_tier",        # Viral / High / Medium / Low
        "days_since_publish",
        "view_velocity",        # views per day since publish
    )
    .filter(F.col("video_id").isNotNull())
)

df_fact_video = add_audit_cols(df_fact_video)
fact_video_count = write_gold_table(
    df_fact_video, "fact_video_performance", ["region_code", "run_date"]
)

print(f"fact_video_performance: {fact_video_count:,} rows")
df_fact_video.printSchema()

fact_video_performance: 668 rows
root
 |-- video_id: string (nullable = true)
 |-- channel_id: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- run_date: string (nullable = true)
 |-- view_count: long (nullable = true)
 |-- like_count: long (nullable = true)
 |-- comment_count: long (nullable = true)
 |-- duration_seconds: long (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- engagement_rate: double (nullable = true)
 |-- virality_score: double (nullable = true)
 |-- like_view_ratio: double (nullable = true)
 |-- comment_view_ratio: double (nullable = true)
 |-- trending_rank: integer (nullable = false)
 |-- virality_tier: string (nullable = false)
 |-- days_since_publish: integer (nullable = true)
 |-- view_velocity: double (nullable = true)
 |-- _gold_created_at: timestamp (nullable = false)
 |-- _run_date: string (nullable = false)



In [0]:
gold_summary = {
    "layer":                     "gold_star_schema",
    "run_date":                  RUN_DATE,
    "processed_at":              datetime.now(timezone.utc).isoformat(),
    # Dimension counts
    "dim_video_count":           dim_video_count,
    "dim_channel_count":         dim_channel_count,
    "dim_category_count":        dim_category_count,
    "dim_region_count":          dim_region_count,
    "dim_date_count":            dim_date_count,
    # Fact counts
    "fact_video_performance_count":  fact_video_count,
    # "fact_comment_sentiment_count":  fact_sentiment_count,
}

#

%md ## 8 · Summary

In [0]:
print("=" * 65)
print("  GOLD STAR SCHEMA — COMPLETE SUMMARY")
print("=" * 65)
print(f"  Run Date                        : {RUN_DATE}")
print()
print("  DIMENSIONS")
print(f"  dim_video                       : {dim_video_count:>8,} rows")
print(f"  dim_channel                     : {dim_channel_count:>8,} rows")
print(f"  dim_category                    : {dim_category_count:>8,} rows")
print(f"  dim_region                      : {dim_region_count:>8,} rows")
print(f"  dim_date                        : {dim_date_count:>8,} rows")
print()
print("  FACTS")
print(f"  fact_video_performance          : {fact_video_count:>8,} rows")

print()
print(f"  Catalog                         : {GOLD_DB}")
print(f"  Gold S3 path                    : {GOLD_PATH}")
print("=" * 65)

dbutils.notebook.exit(json.dumps(gold_summary))

In [0]:
%sql
select distinct region_code from `youtube-capstone-catalog`.gold.dim_region;

region_code
GB
CA
IN
US
AU


In [0]:
display(df_dim_region)

region_code,country_name,continent,_gold_created_at,_run_date
GB,United Kingdom,Europe,2026-04-27T08:25:06.297Z,2026-04-19
CA,Canada,Americas,2026-04-27T08:25:06.297Z,2026-04-19
IN,India,Asia,2026-04-27T08:25:06.297Z,2026-04-19
US,United States,Americas,2026-04-27T08:25:06.297Z,2026-04-19
AU,Australia,Oceania,2026-04-27T08:25:06.297Z,2026-04-19
